# MinkLoc3D Place Recognition Pipeline Tutorial

Demonstrates 3D point cloud place recognition using MinkLoc3D model with FAISS indexing for database retrieval and evaluation metrics calculation.

For the details on how to prepare the data see the [10_preprocess_2025-03-26-mmpr-datasets_mmpr_dataset_1.ipynb](./10_preprocess_2025-03-26-mmpr-datasets_mmpr_dataset_1.ipynb) notebook

In [1]:
from pathlib import Path
import shutil

import torch
import numpy as np
import pandas as pd
# from open3d.web_visualizer import draw
import matplotlib.pyplot as plt
import open3d as o3d
from torch import Tensor, nn
import MinkowskiEngine as ME
import faiss
from tqdm import tqdm

from mslam_calib_transformation import Transform

from opr.models.place_recognition import MinkLoc3D
from opr.pipelines.place_recognition import PlaceRecognitionPipeline


2025-06-22 14:37:55.538 | WARNING  | opr.models.place_recognition.pointmamba:<module>:16 - The 'pointmamba' package is not installed. Please install it manually if neccessary.


## Constants

In [2]:
REPO_ROOT = Path("/home/docker_mmpr/multimodal-place-recognition")

DATA_DIR = REPO_ROOT / "data" / "2025-03-26-mmpr-datasets" / "mmpr_dataset_1" / "mav0"
assert DATA_DIR.exists(), f"Data directory {DATA_DIR} does not exist."

# Lower values = higher resolution but larger memory usage and slower processing
PC_QUANTIZATION_SIZE = 0.05

## DataReader: CSV + Point Cloud Loader

In [3]:
class DataReader:
    def __init__(
            self, csv_file: str | Path, lidar_scans_dir: str | Path, pointcloud_quantization_size: float = 0.1
        ) -> None:
        """Initialize DataReader for pose-timestamped point cloud data.

        Args:
            csv_file (str | Path): Path to the CSV file containing pose and timestamp data.
            lidar_scans_dir (str | Path): Directory containing the lidar scans in PCD format.
            pointcloud_quantization_size (float): Size for quantizing the point cloud coordinates.
                Default is 0.1.
        Raises:
            FileNotFoundError: If the CSV file or lidar scans directory does not exist.
        """
        csv_file = Path(csv_file)
        if not csv_file.exists():
            raise FileNotFoundError(f"CSV file {csv_file} does not exist.")

        self.lidar_scans_dir = Path(lidar_scans_dir)
        if not self.lidar_scans_dir.exists():
            raise FileNotFoundError(f"Lidar scans directory {self.lidar_scans_dir} does not exist.")

        self.df = self.read_csv(csv_file)

        self.scans_list = []
        for scan_id in self.df['lidar_timestamp'].values:
            path = self.lidar_scans_dir / f"{scan_id:018d}.pcd"
            if not path.exists():
                raise FileNotFoundError(f"Missing scan file: {path}")
            self.scans_list.append(path)

        self._pointcloud_quantization_size = pointcloud_quantization_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int) -> dict[str, Tensor]:
        """Get the pose and point cloud data for a given index.

        Args:
            idx (int): Index of the data point to retrieve.
        Returns:
            dict: A dictionary containing:
                - pose (Tensor): The pose as a 7-element tensor [x, y, z, qx, qy, qz, qw].
                - pointcloud_lidar_coords (Tensor): The coordinates of the point cloud as an Nx3 tensor.
                - pointcloud_lidar_feats (Tensor): The features of the point cloud as an Nx1 tensor (intensity or ones).
        Raises:
            IndexError: If the index is out of range.
            ValueError: If the scan file is empty or has an unexpected format.
        """
        if idx < 0 or idx >= len(self.df):
            raise IndexError("Index out of range.")

        pose = self.df[["x", "y", "z", "qx", "qy", "qz", "qw"]].iloc[idx].to_numpy()
        scan_filepath = self.scans_list[idx]
        pc_coords, pc_feats = self.read_scan(scan_filepath)

        output_dict = {
            "pose": Tensor(pose),
            "pointcloud_lidar_coords": Tensor(pc_coords),
            "pointcloud_lidar_feats": Tensor(pc_feats)
        }

        return output_dict

    def collate_fn(self, batch: list[dict[str, Tensor]]) -> dict[str, Tensor]:
        """Collate function to combine a batch of data points into a single dictionary.
        Args:
            batch (list[dict[str, Tensor]]): A list of dictionaries containing pose and point cloud data.
        Returns:
            dict: A dictionary containing:
                - poses (Tensor): Poses as an Nx7 tensor.
                - pointclouds_lidar_coords (Tensor): Point cloud coordinates as an Nx3 tensor.
                - pointclouds_lidar_feats (Tensor): Point cloud features as an Nx1 tensor.
        """
        poses = torch.stack([item['pose'] for item in batch])

        coords_list = [e["pointcloud_lidar_coords"] for e in batch]
        feats_list = [e["pointcloud_lidar_feats"] for e in batch]
        quantized_coords_list = []
        quantized_feats_list = []
        for coords, feats in zip(coords_list, feats_list):
            quantized_coords, quantized_feats = ME.utils.sparse_quantize(
                coordinates=coords,
                features=feats,
                quantization_size=self._pointcloud_quantization_size,
            )
            quantized_coords_list.append(quantized_coords)
            quantized_feats_list.append(quantized_feats)

        return {
            "poses": poses,
            "pointclouds_lidar_coords": ME.utils.batched_coordinates(quantized_coords_list),
            "pointclouds_lidar_feats": torch.cat(quantized_feats_list)
        }

    def read_scan(self, scan_filepath: str | Path) -> tuple[np.ndarray, np.ndarray]:
        """Read a point cloud scan from a file.
        Args:
            scan_filepath (str | Path): Path to the point cloud file.
        Returns:
            tuple: A tuple containing:
                - coordinates (np.ndarray): The coordinates of the point cloud as an Nx3 array.
                - features (np.ndarray): The features of the point cloud as an Nx1 array (intensity or ones).
        Raises:
            ValueError: If the scan file is empty or has an unexpected format.
        """
        scan = o3d.io.read_point_cloud(str(scan_filepath))
        if not scan.has_points():
            raise ValueError(f"Scan file {scan_filepath} is empty or invalid.")
        # Convert to numpy array for easier manipulation
        scan = np.asarray(scan.points)
        coordinates = scan[:, :3]  # Get the first three columns (x, y, z)
        if scan.shape[1] == 3:
            features = np.ones((coordinates.shape[0], 1))
        elif scan.shape[1] == 4:
            features = scan[:, 3:4]  # Get the fourth column (intensity)
        else:
            raise ValueError(f"Unexpected scan format with shape {scan.shape}. Expected 3 or 4 columns.")
        return coordinates, features

    def read_csv(self, filepath: str | Path) -> pd.DataFrame:
        """Read a CSV file containing pose and timestamp data.
        Args:
            filepath (str | Path): Path to the CSV file.
        Returns:
            pd.DataFrame: A DataFrame containing the pose and timestamp data.
        Raises:
            FileNotFoundError: If the CSV file does not exist.
        """
        dtype_mapping = {
            'pose_timestamp': np.int64,
            'lidar_timestamp': np.int64,
            'x': np.float64,
            'y': np.float64,
            'z': np.float64,
            'qx': np.float64,
            'qy': np.float64,
            'qz': np.float64,
            'qw': np.float64,
        }
        df = pd.read_csv(filepath, dtype=dtype_mapping)
        return df


In [4]:
# Database: batched processing for efficient descriptor extraction
database_reader = DataReader(
    csv_file=DATA_DIR / "db_lidar_frames.csv",
    lidar_scans_dir=DATA_DIR / "lidar_joined" / "data",
    pointcloud_quantization_size=PC_QUANTIZATION_SIZE,
)

database_dl = torch.utils.data.DataLoader(
    database_reader,
    batch_size=16,
    shuffle=False,
    collate_fn=database_reader.collate_fn,
    num_workers=4,
    pin_memory=True,
    drop_last=False,
)

# Query: single sample processing (no batching needed)
query_reader = DataReader(
    csv_file=DATA_DIR / "query_lidar_frames.csv",
    lidar_scans_dir=DATA_DIR / "lidar_joined" / "data",
    pointcloud_quantization_size=PC_QUANTIZATION_SIZE,  # Must match database for consistency
)

## Initialize model with weights

Pre-trained MinkLoc3D weights from NCLT dataset:

In [5]:
!wget -O ../data/checkpoints/minkloc3d_nclt.pth \
    https://huggingface.co/OPR-Project/PlaceRecognition-NCLT/resolve/main/minkloc3d_nclt.pth

--2025-06-22 14:37:56--  https://huggingface.co/OPR-Project/PlaceRecognition-NCLT/resolve/main/minkloc3d_nclt.pth
Resolving huggingface.co (huggingface.co)... 18.239.50.49, 18.239.50.103, 18.239.50.16, ...
Connecting to huggingface.co (huggingface.co)|18.239.50.49|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://cdn-lfs-us-1.hf.co/repos/e8/30/e8306844a097b119f688c0cfcf564a9f584f52c28b0d3c5b11e560cb0c3e7eeb/0e8c4d313cab6d463d61cb5501e9de680d5b7a1be363b67e50ea628f11bf8744?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27minkloc3d_nclt.pth%3B+filename%3D%22minkloc3d_nclt.pth%22%3B&Expires=1750606677&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc1MDYwNjY3N319LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy11cy0xLmhmLmNvL3JlcG9zL2U4LzMwL2U4MzA2ODQ0YTA5N2IxMTlmNjg4YzBjZmNmNTY0YTlmNTg0ZjUyYzI4YjBkM2M1YjExZTU2MGNiMGMzZTdlZWIvMGU4YzRkMzEzY2FiNmQ0NjNkNjFjYjU1MDFlOWRlNjgwZDViN2ExYmUzNjNiNjdlNTBlYTYyOGYxMWJmODc0ND

In [6]:
model = MinkLoc3D()  # default model
checkpoint_path = REPO_ROOT / "data" / "checkpoints" / "minkloc3d_nclt.pth"
assert checkpoint_path.exists(), f"Checkpoint file {checkpoint_path} does not exist."
checkpoint = torch.load(checkpoint_path, map_location="cpu")
model.load_state_dict(checkpoint)
model.eval()
if torch.cuda.is_available():
    model = model.cuda()
else:
    print("CUDA is not available, running on CPU.")

## Build FAISS Index from Database Descriptors

In [7]:
# Extract descriptors from all database point clouds
descriptors_list = []
with torch.no_grad():
    for batch in tqdm(database_dl):
        batch = {k: v.to("cuda") for k, v in batch.items()}
        descriptors = model(batch)["final_descriptor"]
        descriptors_list.append(descriptors)
descriptors = torch.cat(descriptors_list, dim=0)
print(f"Descriptors shape: {descriptors.shape}")

# Create L2 distance FAISS index for nearest neighbor search
faiss_index = faiss.IndexFlatL2(descriptors.shape[1])
faiss_index.add(descriptors.cpu().numpy())
faiss.write_index(
    faiss_index,
    str(REPO_ROOT / "data" / "2025-03-26-mmpr-datasets" / "mmpr_dataset_1_database" / "index.faiss")
)

# Copy pose data as track.csv (required by PlaceRecognitionPipeline)
shutil.copy(
    DATA_DIR / "db_lidar_frames.csv",
    REPO_ROOT / "data" / "2025-03-26-mmpr-datasets" / "mmpr_dataset_1_database" / "track.csv"
)

100%|██████████| 20/20 [00:01<00:00, 12.20it/s]

Descriptors shape: torch.Size([313, 256])


PosixPath('/home/docker_mmpr/multimodal-place-recognition/data/2025-03-26-mmpr-datasets/mmpr_dataset_1_database/track.csv')

## Query Processing & Evaluation


In [8]:
pipeline = PlaceRecognitionPipeline(
    database_dir=REPO_ROOT / "data" / "2025-03-26-mmpr-datasets" / "mmpr_dataset_1_database",
    model=model,
    device="cuda",
    pointcloud_quantization_size=PC_QUANTIZATION_SIZE,
)

In [9]:
# Evaluate place recognition accuracy by comparing retrieved vs ground truth poses
translation_errors = []
rotation_errors = []  # angle in radians
for q_idx, query in tqdm(enumerate(query_reader), total=len(query_reader)):
    query = {k: v.to("cuda") for k, v in query.items()}
    results = pipeline.infer(query)

    db_idx = results["idx"]
    db_pose = results["pose"]  # Retrieved database pose
    q_pose = query["pose"].cpu().numpy()  # Ground truth query pose

    # Euclidean distance between positions
    translation_errors.append(np.linalg.norm(q_pose[:3] - db_pose[:3]))
    # Angular distance between quaternions: θ = 2*arccos(|q1·q2|)
    rotation_errors.append(2 * np.arccos(np.abs(np.dot(q_pose[3:], db_pose[3:]))))

translation_errors = np.array(translation_errors)
rotation_errors = np.array(rotation_errors)

100%|██████████| 1630/1630 [00:34<00:00, 47.38it/s]


In [10]:
print(f"Mean translation error: {np.mean(translation_errors):.2f} m")
print(f"Mean rotation error: {np.rad2deg(np.mean(rotation_errors)):.2f} degrees")

print(f"Median translation error: {np.median(translation_errors):.2f} m")
print(f"Median rotation error: {np.rad2deg(np.median(rotation_errors)):.2f} degrees")

Mean translation error: 10.57 m
Mean rotation error: 102.99 degrees
Median translation error: 4.42 m
Median rotation error: 110.29 degrees


In [11]:
distance_threshold = 5.0  # meters

matches_at_5_meters = translation_errors < distance_threshold
recall_at_1 = np.mean(matches_at_5_meters)

print(f"Recall@1 at {distance_threshold:.1f} m: {recall_at_1:.2%}")

print(f"Mean translation error among matches: {np.mean(translation_errors[matches_at_5_meters]):.2f} m")
print(f"Mean rotation error among matches: {np.rad2deg(np.mean(rotation_errors[matches_at_5_meters])):.2f} degrees")

Recall@1 at 5.0 m: 51.66%
Mean translation error among matches: 1.79 m
Mean rotation error among matches: 111.09 degrees


## Future work

1. Make `PlaceRecognitionPipeline` return top-N candidates and their distances to query
2. Make `PlaceRecognitionPipeline` independent on faiss index filename and track csv filename
3. Make `PlaceRecognitionPipeline` work with different column names in track csv 
   (i.e. `["tx", "ty", ...]` or `["norting", "easting", ...]` or anything similar)
4. Visualize the results via `rerun`